# Лучшие параметры embedding по колонкам

Ноутбук читает полный результат перебора `uzal_embedding_results_all_columns.csv` и оставляет по одной лучшей строке на каждую колонку: минимальный `uzal_cost`, соответствующие `tau` и `dimension`.

In [1]:
from pathlib import Path

import pandas as pd
import plotly.express as px

In [2]:
RESULTS_PATH = Path("uzal_embedding_results_all_columns.csv")
SKIPPED_PATH = Path("uzal_embedding_skipped_columns.csv")
BEST_OUTPUT_PATH = Path("uzal_best_choices_by_column.csv")

In [3]:
results = pd.read_csv(RESULTS_PATH)
results.head(), results.shape

(       column  tau  dimension  uzal_cost
 0  Lab1_G1_N1   98          4   1.021868
 1  Lab1_G1_N1   74          5   1.023780
 2  Lab1_G1_N1   95          4   1.029938
 3  Lab1_G1_N1   73          5   1.030567
 4  Lab1_G1_N1   96          4   1.030584,
 (88000, 4))

In [4]:
required_columns = {"column", "tau", "dimension", "uzal_cost"}
missing_columns = required_columns - set(results.columns)
if missing_columns:
    raise ValueError(f"Missing columns in {RESULTS_PATH}: {sorted(missing_columns)}")

all_result_columns = set(results["column"].dropna())
finite_results = results.dropna(subset=["column", "tau", "dimension", "uzal_cost"]).copy()
finite_results["tau"] = finite_results["tau"].astype(int)
finite_results["dimension"] = finite_results["dimension"].astype(int)

print(f"Rows in full results: {len(results)}")
print(f"Columns in full results: {len(all_result_columns)}")
print(f"Rows with finite uzal_cost: {len(finite_results)}")
print(f"Columns with finite uzal_cost: {finite_results['column'].nunique()}")

Rows in full results: 88000
Columns in full results: 88
Rows with finite uzal_cost: 50810
Columns with finite uzal_cost: 73


In [11]:
best_idx = finite_results.groupby("column")["uzal_cost"].idxmin()

best_choices = (
    finite_results.loc[best_idx, ["column", "tau", "dimension", "uzal_cost"]]
    .sort_values("uzal_cost")
    .reset_index(drop=True)
)

pd.set_option("display.max_rows", None)

display(best_choices)

,column,tau,dimension,uzal_cost
0,Lab1_G3_Lm,29,3,-3.300263
1,Lab1_Hpol,95,5,-2.499525
2,Lab1_he,13,10,-2.352875
3,Lab1_dev,9,4,-2.215497
4,Lab1_G3_T638,27,2,-2.011025
5,Lab1_G3_T600,20,3,-1.912052
6,Lab1_G3_T1002,23,2,-1.841676
7,Lab1_PdoNag,57,5,-1.827930
8,Lab1_dPmg,88,9,-1.805453
9,Lab1_Pm_sm_N,95,10,-1.805192


In [12]:
best_choices[best_choices["dimension"] == 1]

,column,tau,dimension,uzal_cost
11,Lab1_G3_V2,58,1,-1.741078
12,Lab1_G3_dPf1,13,1,-1.725502
14,Lab1_G3_Pc3,44,1,-1.630382
22,Lab1_G3_Pc1,37,1,-1.499619
23,Lab1_G3_Pc2,88,1,-1.436029
26,Lab1_G3_T606,27,1,-1.363678
33,Lab1_G3_T1003,55,1,-1.211444
36,Lab1_G3_Pm,24,1,-1.187189
40,Lab1_G3_V1,15,1,-1.113144
41,Lab1_G3_N3,82,1,-1.031962


In [6]:
columns_with_best = set(best_choices["column"])
columns_without_best = sorted(all_result_columns - columns_with_best)

columns_without_best_df = pd.DataFrame(
    {
        "column": columns_without_best,
        "reason": "no_finite_uzal_cost",
    }
)

columns_without_best_df

,column,reason
0,Lab1_G1_T607,no_finite_uzal_cost
1,Lab1_G3_ЗО_СТ,no_finite_uzal_cost
2,Lab1_G3_КВД,no_finite_uzal_cost
3,Lab1_G3_КНД,no_finite_uzal_cost
4,Lab1_G3_Коксование,no_finite_uzal_cost
5,Lab1_G3_ПО_СТ,no_finite_uzal_cost
6,Lab1_G3_Турбина_ГГ,no_finite_uzal_cost
7,Lab1_Kp,no_finite_uzal_cost
8,Lab1_Rc,no_finite_uzal_cost
9,Lab1_TC_T607,no_finite_uzal_cost


In [7]:
best_choices.to_csv(BEST_OUTPUT_PATH, index=False)
columns_without_best_df.to_csv("uzal_columns_without_finite_cost.csv", index=False)

print(f"Best choices: {len(best_choices)} columns")
print(f"Columns without finite uzal_cost: {len(columns_without_best_df)}")
print(f"Saved: {BEST_OUTPUT_PATH}")

Best choices: 73 columns
Columns without finite uzal_cost: 15
Saved: uzal_best_choices_by_column.csv


In [8]:
if SKIPPED_PATH.exists():
    skipped = pd.read_csv(SKIPPED_PATH)
    display(skipped.head(20))
    print(f"Skipped columns: {len(skipped)}")
else:
    skipped = pd.DataFrame(columns=["column", "reason"])
    print("No skipped-columns file found")

,column,reason
0,Lab1_G2_Fнш,not_enough_data_or_constant
1,Lab1_G2_Fма,not_enough_data_or_constant
2,Lab1_G2_Fмн,not_enough_data_or_constant
3,Lab1_G2_Fств,not_enough_data_or_constant
4,Lab1_G3_Маслосистема,not_enough_data_or_constant
5,Lab1_Lmp_Txt_AO,not_enough_data_or_constant
6,Lab1_Lmp_Txt_BEAO,not_enough_data_or_constant
7,Lab1_Lmp_Txt_BK1,not_enough_data_or_constant
8,Lab1_Lmp_Txt_BK2,not_enough_data_or_constant
9,Lab1_TC_VKPGV,not_enough_data_or_constant


Skipped columns: 29


In [ ]:
fig = px.histogram(
    best_choices,
    x="tau",
    nbins=50,
    title="Distribution of selected tau by column",
)
fig.show()

In [ ]:
fig = px.histogram(
    best_choices,
    x="dimension",
    nbins=10,
    title="Distribution of selected embedding dimension by column",
)
fig.show()

In [ ]:
fig = px.bar(
    best_choices.head(30),
    x="column",
    y="uzal_cost",
    color="dimension",
    hover_data=["tau", "dimension"],
    title="Top 30 columns by minimal uzal_cost",
)
fig.update_layout(xaxis_tickangle=-60)
fig.show()